# 学习时间与考试成绩关系分析

本 Notebook 使用 `data.csv` 中的 40 名虚拟学生数据，分析每周学习时间与考试成绩的关系。运行顺序：读取数据 → 数据检查 → 描述性统计 → 相关性分析 → 可视化 → 线性回归 → 结论。

**研究问题：** 学生每周学习时间越长，考试成绩是否通常越高？

提示：请在项目根目录打开本文件，并在首次运行前安装 `requirements.txt` 中列出的依赖。

In [ ]:
# 导入本项目需要的库
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 让图形中的中文在常见 Windows 环境下正常显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = Path('data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('没有找到 data.csv。请从项目根目录打开并运行 analysis.ipynb。')

print(f'数据文件位置：{DATA_PATH.resolve()}')

## 1. 读取数据

数据已随仓库提供，因此整个分析不需要从网络下载任何数据。每一行代表一名学生；`study_hours_per_week` 的单位为小时，`exam_score` 满分为 100 分。

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'数据维度：{df.shape[0]} 行 × {df.shape[1]} 列')
display(df.head(10))

## 2. 数据检查

在建模前检查列名、数据类型、缺失值、重复记录和关键变量的取值范围。这一步可以避免因文件格式错误而得到不可靠的分析结果。

In [ ]:
expected_columns = [
    'student_id',
    'study_hours_per_week',
    'sleep_hours_per_day',
    'attendance_percent',
    'exam_score',
]

assert list(df.columns) == expected_columns, 'CSV 列名或列顺序与项目要求不一致。'
assert df['student_id'].is_unique, 'student_id 中存在重复编号。'
assert df['study_hours_per_week'].between(0, 168).all(), '学习时间存在不合理的值。'
assert df['sleep_hours_per_day'].between(0, 24).all(), '睡眠时间存在不合理的值。'
assert df['attendance_percent'].between(0, 100).all(), '出勤率存在不合理的值。'
assert df['exam_score'].between(0, 100).all(), '考试成绩存在不合理的值。'

print('数据类型：')
display(df.dtypes.to_frame(name='data_type'))

print('各列缺失值数量：')
display(df.isna().sum().to_frame(name='missing_count'))

print(f'完全重复的记录数：{df.duplicated().sum()}')
print('数据检查通过。')

## 3. 描述性统计

下面的表格给出数值变量的样本数、均值、标准差、四分位数和极值，用于概览这份数据的分布。

In [ ]:
numeric_columns = [
    'study_hours_per_week',
    'sleep_hours_per_day',
    'attendance_percent',
    'exam_score',
]

summary = df[numeric_columns].describe().T.round(2)
summary.index.name = 'variable'
display(summary)

## 4. 相关性分析

Pearson 相关系数用于描述两个连续变量之间的线性关系。系数为正表示两个变量一般同向变化；但相关性不等于因果关系。

In [ ]:
correlation_matrix = df[numeric_columns].corr(method='pearson').round(3)
study_score_correlation = correlation_matrix.loc['study_hours_per_week', 'exam_score']

display(correlation_matrix)
print(f'学习时间与考试成绩的 Pearson 相关系数：{study_score_correlation:.3f}')

## 5. 可视化：散点图

每个点代表一名学生。横轴是每周学习时间，纵轴是考试成绩；图形可以帮助我们直观检查是否存在上升趋势或明显异常点。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    df['study_hours_per_week'],
    df['exam_score'],
    color='#2A6FBB',
    edgecolor='white',
    linewidth=0.8,
    s=70,
    alpha=0.9,
)
ax.set_title('每周学习时间与考试成绩')
ax.set_xlabel('每周学习时间（小时）')
ax.set_ylabel('考试成绩（分）')
ax.grid(alpha=0.25)
plt.show()

## 6. 一元线性回归与 R²

使用学习时间预测考试成绩。模型形式为：`考试成绩 = 截距 + 系数 × 学习时间`。R²（决定系数）表示模型在本样本中解释成绩变化的比例。

In [ ]:
X = df[['study_hours_per_week']]
y = df['exam_score']

model = LinearRegression()
model.fit(X, y)

predicted_scores = model.predict(X)
slope = float(model.coef_[0])
intercept = float(model.intercept_)
r_squared = r2_score(y, predicted_scores)

results = pd.DataFrame(
    {
        'metric': ['截距', '学习时间系数', 'R²'],
        'value': [intercept, slope, r_squared],
    }
).round(3)

display(results)
print(f'回归方程：考试成绩 = {intercept:.2f} + {slope:.2f} × 每周学习时间')
print(f'解释：在本样本中，每周学习时间每增加 1 小时，模型预测成绩平均增加约 {slope:.2f} 分。')

In [ ]:
plot_data = df.sort_values('study_hours_per_week')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    df['study_hours_per_week'],
    y,
    color='#2A6FBB',
    edgecolor='white',
    linewidth=0.8,
    s=65,
    label='实际成绩',
)
ax.plot(
    plot_data['study_hours_per_week'],
    model.predict(plot_data[['study_hours_per_week']]),
    color='#D1495B',
    linewidth=2.5,
    label='线性回归线',
)
ax.set_title(f'学习时间与考试成绩的线性回归（R² = {r_squared:.3f}）')
ax.set_xlabel('每周学习时间（小时）')
ax.set_ylabel('考试成绩（分）')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 7. 残差检查

残差是实际成绩与模型预测成绩之差。简单查看残差图可以帮助判断模型误差是否存在明显的系统性趋势。

In [ ]:
residuals = y - predicted_scores

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df['study_hours_per_week'], residuals, color='#4C956C', s=65, alpha=0.9)
ax.axhline(0, color='#333333', linewidth=1, linestyle='--')
ax.set_title('线性回归残差图')
ax.set_xlabel('每周学习时间（小时）')
ax.set_ylabel('残差（实际值 − 预测值）')
ax.grid(alpha=0.25)
plt.show()

print(f'残差均值：{residuals.mean():.3f}')
print(f'残差标准差：{residuals.std():.3f}')

## 8. 结论

本项目在给定的教学数据中观察到学习时间与考试成绩的正线性关系。请结合下面自动生成的结果阅读：

- 若相关系数为正且回归系数为正，说明样本中学习时间较长的学生通常成绩更高。
- R² 只描述这个模型对**当前数据**的拟合程度，并不能直接推广到所有学生。
- 本分析是观察性描述，睡眠、出勤、基础水平和学习方法等变量都可能影响成绩，因此不能由相关或回归直接得出因果结论。

In [ ]:
relationship = '正相关' if study_score_correlation > 0 else '负相关'

print('分析结论')
print('-' * 40)
print(f'学习时间与考试成绩呈 {relationship}（r = {study_score_correlation:.3f}）。')
print(f'线性回归模型的 R² 为 {r_squared:.3f}。')
print('在这份模拟教学数据中，学习时间与成绩存在明显的正向线性关系。')
print('这是一项描述性分析，不能证明学习时间是成绩变化的唯一原因。')